# LegisAnalytica — exportação Gold → PostgreSQL

Recorte de análise: **2025-01-01 a 2025-12-31**. Registros relacionados de outros
anos preservam contexto; não devem entrar automaticamente em indicadores anuais.

1. Execute a Silver e a Gold corrigidas, sem reconstruí-las durante a exportação.
2. Prepare uma base separada com as migrations. Este notebook não aplica DDL.
3. Execute `prepare`: confere schema, dados e chaves existentes sem escrever.
4. Execute `pilot`: carga técnica reduzida por Casa/mês, preservando FKs.
5. Confira reconciliação, espaço por tabela e Disk no painel do Supabase.
6. Execute `full`: atualiza registros e reconcilia contagens, conteúdo e FKs.

Não publique secrets, credenciais nem saídas de conexão. Consulte `README.md`
e `../METODOLOGIA_DADOS.md` antes de carregar uma base existente.


In [ ]:
%pip install psycopg

In [ ]:
# ============================================================
# psycopg v3. Forçamos a implementação Python antes do import
# porque a versão binária já provocou travamento do kernel neste
# ambiente. Requer libpq disponível no runtime.
# Instalação, se necessária, numa célula separada:
# %pip install psycopg
# Após instalar, iniciar uma nova sessão Python.
import os
os.environ["PSYCOPG_IMPL"] = "python"

try:
    import psycopg
except ImportError as exc:
    raise ImportError(
        "Instale 'psycopg' em uma célula separada e reinicie a sessão Python."
    ) from exc

print(f"psycopg {psycopg.__version__}; implementação: {psycopg.pq.__impl__}")

In [ ]:
# ============================================================
# Passo 2 — Configuração e recorte temporal
# ============================================================
# Definição das 26 tabelas Gold, suas colunas, PKs, ordem de
# dependência de FKs (ordem topológica) e filtro temporal.
# As credenciais são lidas do Secret Scope — nunca hardcodeadas.

import re
from datetime import datetime as _dt

# --- Secret Scope ---
SECRET_SCOPE = "legisanalytica_supabase"

# --- Schema Gold no Unity Catalog ---
GOLD_SCHEMA = "workspace.pi_ii_gold"

# --- Tamanho do lote (batch) para inserção ---
BATCH_SIZE = 5000

# --- Recorte temporal (parametrizado) ---
DATA_INICIO = "2025-01-01"
DATA_FIM = "2025-12-31"
ANO_INICIO = _dt.strptime(DATA_INICIO, "%Y-%m-%d").year  # 2025
ANO_FIM = _dt.strptime(DATA_FIM, "%Y-%m-%d").year        # 2025

# --- Limites de armazenamento Supabase Free ---
# A execução começa SEM gravações. Alterar manualmente após revisar:
#   prepare: consulta de conexão/definições; nenhuma inserção
#   pilot: exporta um subconjunto por Casa e mês, com os pais necessários
#   full: exporta as 26 tabelas, com validação final
EXPORT_MODE = "prepare"

PG_SIZE_WARN_MB = 300       # inclui tabelas, índices e objetos do banco
PG_SIZE_ABORT_MB = 350      # margem operacional antes dos 500 MB
WAL_SIZE_WARN_MB = 1200
WAL_SIZE_ABORT_MB = 1400
# Conservador com base na sobra de disco observada no projeto anterior.
# Não substitui a medição do disco físico no painel do Supabase.
DB_PLUS_WAL_ABORT_MB = 1300

if EXPORT_MODE not in {"prepare", "pilot", "full"}:
    raise ValueError("EXPORT_MODE deve ser prepare, pilot ou full")

# --- Configuração das 26 tabelas (ordem topológica de FK) ---
# Cada entrada: (gold_table, colunas, pk_colunas, filter_sql)
# filter_sql = None  → exportar tabela completa (catálogos, dims)
# filter_sql = string → ler subconjunto via spark.sql(filter_sql)

_G = GOLD_SCHEMA  # alias para legibilidade

# Proposições do ano, mais as anteriores referenciadas por votações de 2025.
# Essas proposições mais antigas são apenas pais necessários à integridade;
# NÃO constituem proposições apresentadas em 2025 para indicadores anuais.
_PROPOSICOES_2025_SQL = f"""
SELECT p.* FROM {_G}.proposicao p
WHERE (p.data_apresentacao BETWEEN DATE '{DATA_INICIO}' AND DATE '{DATA_FIM}')
   OR (p.data_apresentacao IS NULL AND p.ano = {ANO_INICIO})
   OR EXISTS (
       SELECT 1 FROM {_G}.votacao_proposicao vp
       JOIN {_G}.votacao v
         ON v.casa = vp.casa AND v.id_votacao = vp.id_votacao
       WHERE vp.casa = p.casa AND vp.id_proposicao = p.id_proposicao
         AND v.data_votacao BETWEEN DATE '{DATA_INICIO}' AND DATE '{DATA_FIM}'
   )
"""

# Votações do recorte de 2025 + votações vinculadas às proposições exportadas.
# Isso preserva a situação de propostas antigas que foram votadas e permite
# recuperar os vínculos de votação do Senado que já existem no Gold.
_VOTACOES_EXPORT_SQL = f"""
SELECT v.* FROM {_G}.votacao v
WHERE (v.data_votacao BETWEEN DATE '{DATA_INICIO}' AND DATE '{DATA_FIM}')
   OR EXISTS (
       SELECT 1 FROM {_G}.votacao_proposicao vp
       INNER JOIN ({_PROPOSICOES_2025_SQL}) p
         ON p.casa = vp.casa AND p.id_proposicao = vp.id_proposicao
       WHERE vp.casa = v.casa AND vp.id_votacao = v.id_votacao
   )
"""

TABLES = [
    # NÍVEL 0 — Catálogos raiz (COMPLETOS)
    ("lote",
     ["id_lote", "entidade_fonte", "localizador", "hash_sha256",
      "processado_em", "coletado_em", "cobertura_inicio", "cobertura_fim"],
     ["id_lote"], None),
    ("casa", ["casa", "nome"], ["casa"], None),
    ("uf", ["uf", "nome"], ["uf"], None),
    ("legislatura", ["numero_legislatura", "inicio", "fim"], ["numero_legislatura"], None),
    ("partido", ["sigla"], ["sigla"], None),
    ("tipo_proposicao", ["sigla_tipo", "descricao"], ["sigla_tipo"], None),

    # NÍVEL 1 (COMPLETOS)
    ("causa_afastamento", ["casa", "codigo_causa", "descricao"], ["casa", "codigo_causa"], None),
    ("parlamentar",
     ["casa", "id_parlamentar", "nome_referencia", "url_foto", "email", "id_lote"],
     ["casa", "id_parlamentar"], None),
    ("tipo_voto", ["casa", "codigo_voto", "descricao"], ["casa", "codigo_voto"], None),
    ("tipo_despesa", ["casa", "codigo_tipo", "descricao"], ["casa", "codigo_tipo"], None),

    # NÍVEL 2
    ("identificador_parlamentar",
     ["casa", "sistema", "codigo_externo", "id_parlamentar"],
     ["casa", "sistema", "codigo_externo"], None),
    ("cadastro_observado",
     ["casa", "id_parlamentar", "id_lote", "partido", "uf"],
     ["casa", "id_parlamentar", "id_lote"], None),
    ("parlamentar_legislatura",
     ["casa", "id_parlamentar", "numero_legislatura", "criterio_inclusao", "id_lote"],
     ["casa", "id_parlamentar", "numero_legislatura"], None),
    ("mandato", ["casa", "id_mandato", "uf", "id_lote"], ["casa", "id_mandato"], None),
    # fornecedor: apenas os referenciados por despesas do recorte
    ("fornecedor", ["casa", "cnpj_cpf", "nome", "id_lote"], ["casa", "cnpj_cpf"],
     f"""SELECT f.* FROM {_G}.fornecedor f
         INNER JOIN (SELECT DISTINCT casa, cnpj_cpf FROM {_G}.despesa
                     WHERE ano >= {ANO_INICIO} AND ano <= {ANO_FIM}
                       AND cnpj_cpf IS NOT NULL) d
         ON f.casa = d.casa AND f.cnpj_cpf = d.cnpj_cpf"""),
    # proposicao: 2025 + pais anteriores necessários às votações de 2025
    ("proposicao",
     ["casa", "id_proposicao", "sigla_tipo", "numero", "ano", "ementa",
      "data_apresentacao", "url_inteiro_teor", "id_lote"],
     ["casa", "id_proposicao"], _PROPOSICOES_2025_SQL),

    # NÍVEL 3
    ("vinculo_mandato",
     ["casa", "id_mandato", "id_parlamentar", "participacao", "id_lote"],
     ["casa", "id_mandato", "id_parlamentar"], None),
    # autoria: INNER JOIN com proposicao do recorte
    ("autoria",
     ["casa", "id_proposicao", "id_autor", "nome_autor", "partido", "uf",
      "ordem_assinatura", "principal", "id_lote"],
     ["casa", "id_proposicao", "id_autor"],
     f"""SELECT a.* FROM {_G}.autoria a
         INNER JOIN ({_PROPOSICOES_2025_SQL}) p
           ON a.casa = p.casa AND a.id_proposicao = p.id_proposicao"""),
    # status_atual: INNER JOIN com proposicao do recorte
    ("status_atual",
     ["casa", "id_proposicao", "data_hora", "sigla_orgao",
      "descricao_tramitacao", "descricao_situacao", "apreciacao", "id_lote"],
     ["casa", "id_proposicao"],
     f"""SELECT s.* FROM {_G}.status_atual s
         INNER JOIN ({_PROPOSICOES_2025_SQL}) p
           ON s.casa = p.casa AND s.id_proposicao = p.id_proposicao"""),
    # votacao: 2025 + votações vinculadas às proposições exportadas
    ("votacao",
     ["casa", "id_votacao", "data_votacao", "sigla_orgao", "aprovacao", "resultado_original",
      "descricao", "votos_sim", "votos_nao", "votos_outros", "id_lote"],
     ["casa", "id_votacao"],
     _VOTACOES_EXPORT_SQL),
    # despesa: filtrar por ano
    ("despesa",
     ["casa", "id_despesa", "id_documento", "id_parlamentar", "ano", "mes", "data_emissao",
      "codigo_tipo", "cnpj_cpf", "id_lote"],
     ["casa", "id_despesa"],
     f"SELECT * FROM {_G}.despesa WHERE ano >= {ANO_INICIO} AND ano <= {ANO_FIM}"),
    # total_publicado: filtrar por ano
    ("total_publicado", ["casa", "ano", "total_proposicoes", "id_lote"], ["casa", "ano"],
     f"SELECT * FROM {_G}.total_publicado WHERE ano >= {ANO_INICIO} AND ano <= {ANO_FIM}"),

    # NÍVEL 4
    ("exercicio",
     ["casa", "id_exercicio", "id_mandato", "id_parlamentar", "inicio", "fim",
      "codigo_causa", "id_lote"],
     ["casa", "id_exercicio"], None),
    # votacao_proposicao: todas as ligações das proposições exportadas
    ("votacao_proposicao",
     ["casa", "id_votacao", "id_proposicao", "id_lote"],
     ["casa", "id_votacao", "id_proposicao"],
     f"""SELECT vp.* FROM {_G}.votacao_proposicao vp
         INNER JOIN ({_PROPOSICOES_2025_SQL}) p
           ON p.casa = vp.casa AND p.id_proposicao = vp.id_proposicao"""),
    # valor_despesa: INNER JOIN com despesa do recorte
    ("valor_despesa",
     ["casa", "id_despesa", "valor_documento", "valor_glosa", "valor_liquido", "valor_restituicao", "id_lote"],
     ["casa", "id_despesa"],
     f"""SELECT vd.* FROM {_G}.valor_despesa vd
         INNER JOIN {_G}.despesa d
           ON vd.casa = d.casa AND vd.id_despesa = d.id_despesa
         WHERE d.ano >= {ANO_INICIO} AND d.ano <= {ANO_FIM}"""),

    # NÍVEL 5
    # voto: votos das votações exportadas
    ("voto",
     ["casa", "id_votacao", "id_parlamentar", "codigo_voto", "partido", "uf", "id_lote"],
     ["casa", "id_votacao", "id_parlamentar"],
     f"""SELECT vt.* FROM {_G}.voto vt
         INNER JOIN ({_VOTACOES_EXPORT_SQL}) v
           ON vt.casa = v.casa AND vt.id_votacao = v.id_votacao"""),
]

print(f"Configuradas {len(TABLES)} tabelas para exportação.")
print(f"Recorte temporal: {DATA_INICIO} a {DATA_FIM} (anos {ANO_INICIO}-{ANO_FIM})")
print(f"Batch size: {BATCH_SIZE} linhas por lote.")
print(f"Secret Scope: {SECRET_SCOPE}")
print(f"Modo de execução: {EXPORT_MODE} (prepare = sem escrita)")
print(f"Limite do banco: warn={PG_SIZE_WARN_MB}MB, abort={PG_SIZE_ABORT_MB}MB")

In [ ]:
# ============================================================
# dbutils é disponibilizado automaticamente no notebook Databricks.
# Não criar instância de DBUtils: usar o objeto integrado ao notebook.
# A JDBC URL é convertida removendo apenas o prefixo "jdbc:".

if "dbutils" not in globals():
    raise RuntimeError("Execute este notebook no Databricks, com dbutils disponível.")


def get_connection():
    jdbc_url = dbutils.secrets.get(scope=SECRET_SCOPE, key="jdbc_url")
    user = dbutils.secrets.get(scope=SECRET_SCOPE, key="user")
    password = dbutils.secrets.get(scope=SECRET_SCOPE, key="password")

    # Mantém database e query params (inclusive ?sslmode=require).
    uri = jdbc_url.removeprefix("jdbc:")
    if not uri.startswith("postgresql://"):
        raise ValueError("jdbc_url precisa conter uma URL JDBC PostgreSQL válida.")

    return psycopg.connect(
        conninfo=uri,
        user=user,
        password=password,
        sslmode="require",
        connect_timeout=20,
        autocommit=False,
    )


print("Testando conexão com Supabase...")
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user")
        banco, usuario = cur.fetchone()
print(f"Conexão OK. Banco: {banco}; usuário: {usuario}")

In [ ]:
# ============================================================
# O PostgreSQL mantém WAL independente do tamanho das tabelas.
# O total (banco + WAL) NÃO equivale à ocupação física total:
# acompanhar a métrica Disk no painel do Supabase durante a carga.

import time


def build_insert_sql(table_name, columns, pk_columns):
    # Nomes de tabelas e colunas vêm apenas da configuração estática TABLES.
    cols = ", ".join(columns)
    placeholders = ", ".join(["%s"] * len(columns))
    pk_cols = ", ".join(pk_columns)
    update_columns = [column for column in columns if column not in pk_columns]
    if not update_columns:
        return (f"INSERT INTO public.{table_name} ({cols}) VALUES ({placeholders}) "
                f"ON CONFLICT ({pk_cols}) DO NOTHING")
    assignments = ", ".join(f"{column} = EXCLUDED.{column}" for column in update_columns)
    current = ", ".join(f"atual.{column}" for column in update_columns)
    incoming = ", ".join(f"EXCLUDED.{column}" for column in update_columns)
    return (f"INSERT INTO public.{table_name} AS atual ({cols}) VALUES ({placeholders}) "
            f"ON CONFLICT ({pk_cols}) DO UPDATE SET {assignments} "
            f"WHERE ROW({current}) IS DISTINCT FROM ROW({incoming})")



def scalar(conn, sql):
    with conn.cursor() as cur:
        cur.execute(sql)
        return cur.fetchone()[0]


def storage_metrics(conn):
    # Falha de medição interrompe a importação (não ignora o erro).
    # float(): pg_ls_waldir devolve numeric (Decimal) no psycopg e
    # Decimal + float levantaria TypeError na soma.
    db_mb = float(scalar(conn, "SELECT pg_database_size(current_database())")) / 1048576
    wal_mb = float(scalar(conn, "SELECT COALESCE(SUM(size), 0) FROM pg_ls_waldir()")) / 1048576
    return db_mb, wal_mb


def check_storage(conn):
    db_mb, wal_mb = storage_metrics(conn)
    combined = db_mb + wal_mb
    print(f"    Armazenamento: banco={db_mb:.1f} MB, WAL={wal_mb:.1f} MB")
    if (db_mb >= PG_SIZE_ABORT_MB or wal_mb >= WAL_SIZE_ABORT_MB
            or combined >= DB_PLUS_WAL_ABORT_MB):
        raise RuntimeError(
            "Limiar de segurança de armazenamento alcançado. "
            "Interrompa e confira o disco físico no painel Supabase."
        )
    if db_mb >= PG_SIZE_WARN_MB or wal_mb >= WAL_SIZE_WARN_MB:
        print("    ATENÇÃO: armazenamento próximo dos limites definidos.")
    return db_mb, wal_mb


def export_table(gold_table, columns, pk_columns, conn, df,
                 batch_size=BATCH_SIZE):
    start = time.time()
    gold_count = df.count()
    pg_before = scalar(conn, f"SELECT COUNT(*) FROM public.{gold_table}")
    if not gold_count:
        print(f"  {gold_table}: fonte vazia; banco já contém {pg_before} registros.")
        return dict(table=gold_table, gold_count=0, pg_inserted=0,
                    pg_final=pg_before, elapsed_s=0.0, status="source_empty")

    check_storage(conn)
    insert_sql = build_insert_sql(gold_table, columns, pk_columns)
    inserted_total = 0
    batch_number = 0
    batch = []

    def write_batch(rows):
        nonlocal inserted_total, batch_number
        check_storage(conn)
        batch_number += 1
        try:
            with conn.cursor() as cur:
                cur.executemany(insert_sql, rows)
                inserted_total += max(0, cur.rowcount)
            conn.commit()
            # Verificação APÓS commit; interrompe antes do próximo lote.
            check_storage(conn)
        except Exception as exc:
            conn.rollback()
            raise RuntimeError(
                f"Falha na tabela {gold_table}, lote {batch_number}. "
                "Importação interrompida; lotes anteriores permanecem no banco."
            ) from exc

    # Sem repartition/shuffle adicional: streaming por partição de origem.
    for row in df.select(*columns).toLocalIterator():
        batch.append(tuple(row[col] for col in columns))
        if len(batch) == batch_size:
            write_batch(batch)
            batch = []
            if batch_number % 10 == 0:
                print(f"    {gold_table}: {batch_number} lotes processados")
    if batch:
        write_batch(batch)

    pg_after = scalar(conn, f"SELECT COUNT(*) FROM public.{gold_table}")
    status = "ok" if pg_after == gold_count else "count_mismatch"
    print(f"  {gold_table}: Gold={gold_count:,}; PostgreSQL={pg_after:,}; "
          f"inseridos={inserted_total:,}; status={status}")
    if status != "ok":
        raise RuntimeError(
            f"Contagens divergentes em {gold_table}: fonte={gold_count}, destino={pg_after}. "
            "Há chaves fora do recorte. Revise o destino; a carga não apaga dados antigos."
        )
    return dict(table=gold_table, gold_count=gold_count,
                pg_inserted=inserted_total, pg_final=pg_after,
                elapsed_s=round(time.time() - start, 1), status=status)

In [ ]:
# A criação/alteração do schema é manual e anterior à carga.
# Consulte ../supabase/migrations/README.md. Este notebook não executa DDL.
print("Schema será conferido no preflight; nenhuma migration será executada.")

In [ ]:
# Conferência antes da escrita (preflight).
# Juntamos os erros para mostrar o que precisa ser corrigido antes de carregar.
# PK identifica um registro; FK liga esse registro a uma linha de outra tabela.

from pyspark.sql import functions as F

# Colunas NOT NULL por tabela, conforme a migration aplicada
NOT_NULL = {
    "lote": ["id_lote", "entidade_fonte", "localizador", "hash_sha256", "processado_em"],
    "casa": ["casa", "nome"],
    "uf": ["uf", "nome"],
    "legislatura": ["numero_legislatura", "inicio", "fim"],
    "partido": ["sigla"],
    "tipo_proposicao": ["sigla_tipo"],
    "causa_afastamento": ["casa", "codigo_causa", "descricao"],
    "parlamentar": ["casa", "id_parlamentar", "nome_referencia", "id_lote"],
    "tipo_voto": ["casa", "codigo_voto"],
    "tipo_despesa": ["casa", "codigo_tipo", "descricao"],
    "identificador_parlamentar": ["casa", "sistema", "codigo_externo"],
    "cadastro_observado": ["casa", "id_parlamentar", "id_lote"],
    "parlamentar_legislatura": ["casa", "id_parlamentar", "numero_legislatura", "criterio_inclusao", "id_lote"],
    "mandato": ["casa", "id_mandato", "uf", "id_lote"],
    "fornecedor": ["casa", "cnpj_cpf", "nome", "id_lote"],
    "proposicao": ["casa", "id_proposicao", "sigla_tipo", "id_lote"],
    "vinculo_mandato": ["casa", "id_mandato", "id_parlamentar", "participacao", "id_lote"],
    "autoria": ["casa", "id_proposicao", "id_autor", "principal", "id_lote"],
    "status_atual": ["casa", "id_proposicao", "id_lote"],
    "votacao": ["casa", "id_votacao", "id_lote"],
    "despesa": ["casa", "id_despesa", "ano", "mes", "id_lote"],
    "total_publicado": ["casa", "ano", "total_proposicoes", "id_lote"],
    "exercicio": ["casa", "id_exercicio", "id_mandato", "id_parlamentar", "inicio", "id_lote"],
    "votacao_proposicao": ["casa", "id_votacao", "id_proposicao", "id_lote"],
    "valor_despesa": ["casa", "id_despesa", "id_lote"],
    "voto": ["casa", "id_votacao", "id_parlamentar", "id_lote"],
}

# Os mesmos DataFrames são usados no preflight, na escrita e na reconciliação.
SUBSETS = {}
for tabela, _, _, filtro_sql in TABLES:
    if filtro_sql:
        SUBSETS[tabela] = spark.sql(filtro_sql)
    else:
        SUBSETS[tabela] = spark.table(f"{GOLD_SCHEMA}.{tabela}")

if EXPORT_MODE == "pilot":
    from pyspark.sql.window import Window

    def amostra_por_grupo(df, grupos, chave, tamanho):
        janela = Window.partitionBy(*grupos).orderBy(F.sha2(F.col(chave), 256))
        return df.withColumn("_ordem_piloto", F.row_number().over(janela)).where(
            F.col("_ordem_piloto") <= tamanho
        ).drop("_ordem_piloto")

    # O piloto testa a carga; seus números não representam a população inteira.
    SUBSETS["despesa"] = amostra_por_grupo(
        SUBSETS["despesa"], ["casa", "ano", "mes"], "id_despesa", 5
    )
    SUBSETS["votacao"] = amostra_por_grupo(
        SUBSETS["votacao"], ["casa"], "id_votacao", 5
    )
    chaves_votacoes = ["casa", "id_votacao"]
    chaves_proposicoes = ["casa", "id_proposicao"]
    chaves_despesas = ["casa", "id_despesa"]
    votacoes_piloto = SUBSETS["votacao"].select(*chaves_votacoes)

    # left_semi mantém só as linhas da esquerda que têm correspondência à direita.
    ligacoes = SUBSETS["votacao_proposicao"].join(
        votacoes_piloto, chaves_votacoes, "left_semi"
    )
    proposicoes_das_votacoes = SUBSETS["proposicao"].join(
        ligacoes.select(*chaves_proposicoes), chaves_proposicoes, "left_semi"
    )
    proposicoes_amostradas = amostra_por_grupo(
        SUBSETS["proposicao"], ["casa"], "id_proposicao", 5
    )
    # Incluímos a proposta de cada votação escolhida, mesmo fora da amostra inicial.
    SUBSETS["proposicao"] = (
        proposicoes_amostradas.unionByName(proposicoes_das_votacoes)
        .dropDuplicates(chaves_proposicoes)
    )
    proposicoes_piloto = SUBSETS["proposicao"].select(*chaves_proposicoes)

    for entidade in ["autoria", "status_atual", "votacao_proposicao"]:
        SUBSETS[entidade] = SUBSETS[entidade].join(
            proposicoes_piloto, chaves_proposicoes, "left_semi"
        )
    SUBSETS["votacao_proposicao"] = SUBSETS["votacao_proposicao"].join(
        votacoes_piloto, chaves_votacoes, "left_semi"
    )
    SUBSETS["voto"] = SUBSETS["voto"].join(
        votacoes_piloto, chaves_votacoes, "left_semi"
    )
    despesas_piloto = SUBSETS["despesa"].select(*chaves_despesas)
    SUBSETS["valor_despesa"] = SUBSETS["valor_despesa"].join(
        despesas_piloto, chaves_despesas, "left_semi"
    )
    fornecedores_piloto = SUBSETS["despesa"].select("casa", "cnpj_cpf")
    SUBSETS["fornecedor"] = SUBSETS["fornecedor"].join(
        fornecedores_piloto, ["casa", "cnpj_cpf"], "left_semi"
    )

preflight_errors = []

# 1) Campos obrigatórios: NOT NULL não permite valor ausente.
for t, cols in NOT_NULL.items():
    row = SUBSETS[t].agg(
        *[F.sum(F.col(c).isNull().cast("long")).alias(c) for c in cols]
    ).first()
    for c in cols:
        if row[c]:
            preflight_errors.append(
                f"{t}.{c}: {int(row[c])} NULL(s) — viola NOT NULL da migration")

# 2) Cada chave primária precisa identificar uma única linha.
for gold_table, columns, pk_columns, _ in TABLES:
    df = SUBSETS[gold_table]
    n, n_pk = df.count(), df.select(*pk_columns).distinct().count()
    if n != n_pk:
        preflight_errors.append(
            f"{gold_table}: {n - n_pk} duplicidade(s) de PK no subset")

# 3) lote — UNIQUE (entidade_fonte), UNIQUE (hash_sha256), CHAR(64)
_lote = SUBSETS["lote"]
_n_lote = _lote.count()
for _c in ("entidade_fonte", "hash_sha256"):
    if _lote.select(_c).distinct().count() != _n_lote:
        preflight_errors.append(f"lote.{_c}: duplicidade viola UNIQUE da migration")
_n_long = _lote.where(F.length("hash_sha256") > 64).count()
if _n_long:
    preflight_errors.append(f"lote.hash_sha256: {_n_long} valor(es) acima de CHAR(64)")

# 4) Limites VARCHAR da migration
for t, c, lim in (("parlamentar", "casa", 10), ("mandato", "uf", 2),
                  ("identificador_parlamentar", "sistema", 10), ("voto", "uf", 2)):
    _n = SUBSETS[t].where(F.length(c) > lim).count()
    if _n:
        preflight_errors.append(f"{t}.{c}: {_n} valor(es) excedem VARCHAR({lim})")

# 5) Schema public do Supabase: tabelas e colunas esperadas
with get_connection() as conn:
    for gold_table, columns, pk_columns, _ in TABLES:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT column_name FROM information_schema.columns "
                "WHERE table_schema = 'public' AND table_name = %s",
                (gold_table,),
            )
            pg_cols = {r[0] for r in cur.fetchall()}
        if not pg_cols:
            preflight_errors.append(f"public.{gold_table}: tabela não encontrada no Supabase")
            continue
        for c in sorted(set(columns) - pg_cols):
            preflight_errors.append(f"public.{gold_table}.{c}: coluna ausente no PostgreSQL")
        for c in sorted(pg_cols - set(columns)):
            preflight_errors.append(f"public.{gold_table}.{c}: coluna extra no PostgreSQL")

# Verifica as FKs declaradas no destino usando os próprios subconjuntos da carga.
with get_connection() as conn:
    for tabela, _, chaves, _ in TABLES:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT atributo.attname FROM pg_index indice
                CROSS JOIN LATERAL unnest(indice.indkey) WITH ORDINALITY AS chave(numero, ordem)
                JOIN pg_attribute atributo ON atributo.attrelid = indice.indrelid AND atributo.attnum = chave.numero
                WHERE indice.indrelid = to_regclass(%s) AND indice.indisprimary
                ORDER BY chave.ordem
            """, (f"public.{tabela}",))
            chaves_destino = [row[0] for row in cur.fetchall()]
        if chaves_destino != chaves:
            preflight_errors.append(f"{tabela}: PK do destino difere da configuração da carga")
    with conn.cursor() as cur:
        cur.execute("""
            SELECT filho.relname, array_agg(cf.attname ORDER BY chave.ordem),
                   pai.relname, array_agg(cp.attname ORDER BY chave.ordem)
            FROM pg_constraint fk
            JOIN pg_class filho ON filho.oid = fk.conrelid
            JOIN pg_namespace ns ON ns.oid = filho.relnamespace
            JOIN pg_class pai ON pai.oid = fk.confrelid
            CROSS JOIN LATERAL unnest(fk.conkey, fk.confkey) WITH ORDINALITY AS chave(filha, pai, ordem)
            JOIN pg_attribute cf ON cf.attrelid = filho.oid AND cf.attnum = chave.filha
            JOIN pg_attribute cp ON cp.attrelid = pai.oid AND cp.attnum = chave.pai
            WHERE fk.contype = 'f' AND ns.nspname = 'public'
            GROUP BY fk.oid, filho.relname, pai.relname
        """)
        relacionamentos = cur.fetchall()
# A migration inicial declara 60 FKs; não aceitar um schema sem essas proteções.
relacionamentos_carga = [r for r in relacionamentos if r[0] in SUBSETS and r[2] in SUBSETS]
if len(relacionamentos_carga) < 60:
    preflight_errors.append("Schema incompleto: menos de 60 FKs entre as tabelas da carga")
for filha, colunas_filhas, pai, colunas_pai in relacionamentos_carga:
    # Uma FK opcional com campo nulo não aponta para outra linha e é ignorada.
    validas = F.lit(True)
    for coluna in colunas_filhas:
        validas = validas & F.col(coluna).isNotNull()
    pais = SUBSETS[pai].select(*[F.col(p).alias(f) for f, p in zip(colunas_filhas, colunas_pai)])
    orfaos = SUBSETS[filha].where(validas).join(pais, colunas_filhas, "left_anti").count()
    if orfaos:
        preflight_errors.append(f"{filha} → {pai}: {orfaos} órfãos antes da carga")

if SUBSETS["despesa"].where("mes NOT BETWEEN 1 AND 12 OR ano <> 2025").limit(1).count():
    preflight_errors.append("Despesas fora da competência de 2025 ou com mês inválido")
valores_ausentes = SUBSETS["despesa"].join(
    SUBSETS["valor_despesa"].where("valor_liquido IS NOT NULL").select("casa", "id_despesa"),
    ["casa", "id_despesa"], "left_anti"
).count()
if valores_ausentes:
    preflight_errors.append(f"{valores_ausentes} despesas sem valor líquido conhecido")

# Atualizar registros existentes é seguro somente se o destino não tiver chaves extras.
# Uma tabela de chaves por vez fica na memória do driver, nunca os registros completos.
if not preflight_errors:
    with get_connection() as conn:
        for tabela, _, chaves, _ in TABLES:
            chaves_fonte = {tuple(row) for row in SUBSETS[tabela].select(*chaves).toLocalIterator()}
            extras = 0
            with conn.cursor(name=f"chaves_{tabela}") as cur:
                cur.execute(f"SELECT {', '.join(chaves)} FROM public.{tabela}")
                for row in cur:
                    if tuple(row) not in chaves_fonte:
                        extras += 1
            if extras:
                preflight_errors.append(f"{tabela}: {extras} chaves no destino fora da fonte; use uma base separada para a recarga")

print(f"Pre-flight: {len(SUBSETS)} subsets validados; "
      f"{len(preflight_errors)} erro(s) encontrado(s).")
if preflight_errors:
    for e in preflight_errors:
        print("  FALHA:", e)
    raise RuntimeError("Pre-flight reprovado — a exportação NÃO foi iniciada.")
print("Pre-flight APROVADO: NOT NULL, PKs, UNIQUE, VARCHAR, FKs, valores e chaves do destino verificados.")

In [ ]:
# ============================================================
# A escrita está DESLIGADA por padrão.
# prepare: só valida; pilot: exporta subconjuntos com seus relacionamentos;
# full: exporta as 26 tabelas e habilita a validação integral.

results = []
export_completed = False

if EXPORT_MODE == "prepare":
    print("Modo prepare: NENHUMA escrita executada no Supabase.")
else:
    selected = TABLES
    print(f"Iniciando {EXPORT_MODE}: {len(selected)} tabela(s), período {DATA_INICIO} a {DATA_FIM}.")
    with get_connection() as conn:
        try:
            for i, (gold_table, columns, pk_columns, filter_sql) in enumerate(selected, 1):
                print(f"[{i}/{len(selected)}] {gold_table}")
                results.append(export_table(gold_table, columns, pk_columns, conn, SUBSETS[gold_table]))
        except BaseException:
            conn.rollback()
            raise
    export_completed = True
    print(f"Execução {EXPORT_MODE} concluída: {len(results)} tabela(s).")

In [ ]:
# Contagem e FKs não detectam conteúdo desatualizado. O fingerprint inclui todos os campos.
import hashlib
import json
from datetime import date, datetime
from decimal import Decimal


def valor_canonico(valor):
    """Coloca datas e decimais no mesmo formato na origem e no destino."""
    if isinstance(valor, Decimal):
        if valor == 0:
            return "0"
        return format(valor.normalize(), "f")
    if isinstance(valor, (date, datetime)):
        return valor.isoformat()
    return valor


def fingerprint_linhas(linhas):
    """Retorna a quantidade de linhas e um resumo calculado de seu conteúdo."""
    quantidade = 0
    soma_hashes = 0
    limite_hash = 2 ** 256
    for linha in linhas:
        valores = [valor_canonico(valor) for valor in linha]
        serializado = json.dumps(valores, ensure_ascii=False, separators=(",", ":"))
        hash_linha = hashlib.sha256(serializado.encode("utf-8")).digest()
        hash_como_numero = int.from_bytes(hash_linha, "big")
        # A soma aceita outra ordem de leitura e ainda conta as linhas repetidas.
        soma_hashes = (soma_hashes + hash_como_numero) % limite_hash
        quantidade += 1
    return quantidade, f"{soma_hashes:064x}"


validation_results = []
all_match = False
if not export_completed:
    print("Reconciliação adiada: nenhuma carga executada.")
else:
    with get_connection() as conn:
        for tabela, colunas, _, _ in TABLES:
            fonte = fingerprint_linhas(SUBSETS[tabela].select(*colunas).toLocalIterator())
            # Cursor no servidor evita carregar uma tabela inteira na memória local.
            with conn.cursor(name=f"validar_{tabela}") as cur:
                cur.execute(f"SELECT {', '.join(colunas)} FROM public.{tabela}")
                destino = fingerprint_linhas(cur)
            igual = fonte == destino
            validation_results.append(dict(table=tabela, source=fonte, target=destino, match=igual))
            print(f"  {tabela}: fonte={fonte[0]}, destino={destino[0]}, conteúdo={'OK' if igual else 'DIVERGENTE'}")
    all_match = all(resultado["match"] for resultado in validation_results)
    if not all_match:
        raise RuntimeError("Reconciliação de conteúdo reprovada. Não disponibilize esta carga para análise.")

In [ ]:
# ============================================================
# Passo 7 — Validar integridade referencial (FKs)
# ============================================================
# Verifica todas as FKs no PostgreSQL, contando registros órfãos.
# FKs com colunas NULLABLE usam a condição WHERE col IS NOT NULL
# para ignorar NULLs (comportamento MATCH SIMPLE).
# As tabelas filhas foram filtradas por INNER JOIN com os pais no
# recorte 2025, garantindo zero órfãos por construção.

# Lista de verificações: (descrição, SQL de contagem de órfãos)
FK_CHECKS = [
    # NÍVEL 1 → catálogos
    ("causa_afastamento → casa",
     "SELECT COUNT(*) FROM causa_afastamento c LEFT JOIN casa ON c.casa = casa.casa WHERE casa.casa IS NULL"),
    ("parlamentar → casa",
     "SELECT COUNT(*) FROM parlamentar p LEFT JOIN casa ON p.casa = casa.casa WHERE casa.casa IS NULL"),
    ("parlamentar → lote",
     "SELECT COUNT(*) FROM parlamentar p LEFT JOIN lote ON p.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("tipo_voto → casa",
     "SELECT COUNT(*) FROM tipo_voto t LEFT JOIN casa ON t.casa = casa.casa WHERE casa.casa IS NULL"),
    ("tipo_despesa → casa",
     "SELECT COUNT(*) FROM tipo_despesa t LEFT JOIN casa ON t.casa = casa.casa WHERE casa.casa IS NULL"),

    # NÍVEL 2
    ("identificador_parlamentar → parlamentar (NULL ok)",
     "SELECT COUNT(*) FROM identificador_parlamentar i LEFT JOIN parlamentar p ON i.casa = p.casa AND i.id_parlamentar = p.id_parlamentar WHERE i.id_parlamentar IS NOT NULL AND p.id_parlamentar IS NULL"),
    ("cadastro_observado → parlamentar",
     "SELECT COUNT(*) FROM cadastro_observado c LEFT JOIN parlamentar p ON c.casa = p.casa AND c.id_parlamentar = p.id_parlamentar WHERE p.id_parlamentar IS NULL"),
    ("cadastro_observado → lote",
     "SELECT COUNT(*) FROM cadastro_observado c LEFT JOIN lote ON c.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("cadastro_observado → partido (NULL ok)",
     "SELECT COUNT(*) FROM cadastro_observado c LEFT JOIN partido ON c.partido = partido.sigla WHERE c.partido IS NOT NULL AND partido.sigla IS NULL"),
    ("cadastro_observado → uf (NULL ok)",
     "SELECT COUNT(*) FROM cadastro_observado c LEFT JOIN uf ON c.uf = uf.uf WHERE c.uf IS NOT NULL AND uf.uf IS NULL"),
    ("parlamentar_legislatura → parlamentar",
     "SELECT COUNT(*) FROM parlamentar_legislatura pl LEFT JOIN parlamentar p ON pl.casa = p.casa AND pl.id_parlamentar = p.id_parlamentar WHERE p.id_parlamentar IS NULL"),
    ("parlamentar_legislatura → legislatura",
     "SELECT COUNT(*) FROM parlamentar_legislatura pl LEFT JOIN legislatura ON pl.numero_legislatura = legislatura.numero_legislatura WHERE legislatura.numero_legislatura IS NULL"),
    ("parlamentar_legislatura → lote",
     "SELECT COUNT(*) FROM parlamentar_legislatura pl LEFT JOIN lote ON pl.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("mandato → casa",
     "SELECT COUNT(*) FROM mandato m LEFT JOIN casa ON m.casa = casa.casa WHERE casa.casa IS NULL"),
    ("mandato → uf",
     "SELECT COUNT(*) FROM mandato m LEFT JOIN uf ON m.uf = uf.uf WHERE uf.uf IS NULL"),
    ("mandato → lote",
     "SELECT COUNT(*) FROM mandato m LEFT JOIN lote ON m.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("fornecedor → casa",
     "SELECT COUNT(*) FROM fornecedor f LEFT JOIN casa ON f.casa = casa.casa WHERE casa.casa IS NULL"),
    ("fornecedor → lote",
     "SELECT COUNT(*) FROM fornecedor f LEFT JOIN lote ON f.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("proposicao → tipo_proposicao",
     "SELECT COUNT(*) FROM proposicao p LEFT JOIN tipo_proposicao ON p.sigla_tipo = tipo_proposicao.sigla_tipo WHERE tipo_proposicao.sigla_tipo IS NULL"),
    ("proposicao → lote",
     "SELECT COUNT(*) FROM proposicao p LEFT JOIN lote ON p.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),

    # NÍVEL 3
    ("vinculo_mandato → mandato",
     "SELECT COUNT(*) FROM vinculo_mandato v LEFT JOIN mandato m ON v.casa = m.casa AND v.id_mandato = m.id_mandato WHERE m.id_mandato IS NULL"),
    ("vinculo_mandato → parlamentar",
     "SELECT COUNT(*) FROM vinculo_mandato v LEFT JOIN parlamentar p ON v.casa = p.casa AND v.id_parlamentar = p.id_parlamentar WHERE p.id_parlamentar IS NULL"),
    ("vinculo_mandato → lote",
     "SELECT COUNT(*) FROM vinculo_mandato v LEFT JOIN lote ON v.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("autoria → proposicao",
     "SELECT COUNT(*) FROM autoria a LEFT JOIN proposicao p ON a.casa = p.casa AND a.id_proposicao = p.id_proposicao WHERE p.id_proposicao IS NULL"),
    ("autoria → lote",
     "SELECT COUNT(*) FROM autoria a LEFT JOIN lote ON a.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("autoria → partido (NULL ok)",
     "SELECT COUNT(*) FROM autoria a LEFT JOIN partido ON a.partido = partido.sigla WHERE a.partido IS NOT NULL AND partido.sigla IS NULL"),
    ("autoria → uf (NULL ok)",
     "SELECT COUNT(*) FROM autoria a LEFT JOIN uf ON a.uf = uf.uf WHERE a.uf IS NOT NULL AND uf.uf IS NULL"),
    ("status_atual → proposicao",
     "SELECT COUNT(*) FROM status_atual s LEFT JOIN proposicao p ON s.casa = p.casa AND s.id_proposicao = p.id_proposicao WHERE p.id_proposicao IS NULL"),
    ("status_atual → lote",
     "SELECT COUNT(*) FROM status_atual s LEFT JOIN lote ON s.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("votacao → lote",
     "SELECT COUNT(*) FROM votacao v LEFT JOIN lote ON v.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("despesa → tipo_despesa (NULL ok)",
     "SELECT COUNT(*) FROM despesa d LEFT JOIN tipo_despesa t ON d.casa = t.casa AND d.codigo_tipo = t.codigo_tipo WHERE d.codigo_tipo IS NOT NULL AND t.codigo_tipo IS NULL"),
    ("despesa → fornecedor (NULL ok)",
     "SELECT COUNT(*) FROM despesa d LEFT JOIN fornecedor f ON d.casa = f.casa AND d.cnpj_cpf = f.cnpj_cpf WHERE d.cnpj_cpf IS NOT NULL AND f.cnpj_cpf IS NULL"),
    ("despesa → lote",
     "SELECT COUNT(*) FROM despesa d LEFT JOIN lote ON d.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("total_publicado → lote",
     "SELECT COUNT(*) FROM total_publicado t LEFT JOIN lote ON t.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),

    # NÍVEL 4
    ("exercicio → vinculo_mandato",
     "SELECT COUNT(*) FROM exercicio e LEFT JOIN vinculo_mandato v ON e.casa = v.casa AND e.id_mandato = v.id_mandato AND e.id_parlamentar = v.id_parlamentar WHERE v.id_mandato IS NULL"),
    ("exercicio → causa_afastamento (NULL ok)",
     "SELECT COUNT(*) FROM exercicio e LEFT JOIN causa_afastamento c ON e.casa = c.casa AND e.codigo_causa = c.codigo_causa WHERE e.codigo_causa IS NOT NULL AND c.codigo_causa IS NULL"),
    ("exercicio → lote",
     "SELECT COUNT(*) FROM exercicio e LEFT JOIN lote ON e.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("votacao_proposicao → votacao",
     "SELECT COUNT(*) FROM votacao_proposicao vp LEFT JOIN votacao v ON vp.casa = v.casa AND vp.id_votacao = v.id_votacao WHERE v.id_votacao IS NULL"),
    ("votacao_proposicao → proposicao",
     "SELECT COUNT(*) FROM votacao_proposicao vp LEFT JOIN proposicao p ON vp.casa = p.casa AND vp.id_proposicao = p.id_proposicao WHERE p.id_proposicao IS NULL"),
    ("votacao_proposicao → lote",
     "SELECT COUNT(*) FROM votacao_proposicao vp LEFT JOIN lote ON vp.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
    ("valor_despesa → despesa",
     "SELECT COUNT(*) FROM valor_despesa vd LEFT JOIN despesa d ON vd.casa = d.casa AND vd.id_despesa = d.id_despesa WHERE d.id_despesa IS NULL"),
    ("valor_despesa → lote",
     "SELECT COUNT(*) FROM valor_despesa vd LEFT JOIN lote ON vd.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),

    # NÍVEL 5
    ("voto → votacao",
     "SELECT COUNT(*) FROM voto v LEFT JOIN votacao ON v.casa = votacao.casa AND v.id_votacao = votacao.id_votacao WHERE votacao.id_votacao IS NULL"),
    ("voto → parlamentar",
     "SELECT COUNT(*) FROM voto v LEFT JOIN parlamentar p ON v.casa = p.casa AND v.id_parlamentar = p.id_parlamentar WHERE p.id_parlamentar IS NULL"),
    ("voto → tipo_voto (NULL ok)",
     "SELECT COUNT(*) FROM voto v LEFT JOIN tipo_voto t ON v.casa = t.casa AND v.codigo_voto = t.codigo_voto WHERE v.codigo_voto IS NOT NULL AND t.codigo_voto IS NULL"),
    ("voto → lote",
     "SELECT COUNT(*) FROM voto v LEFT JOIN lote ON v.id_lote = lote.id_lote WHERE lote.id_lote IS NULL"),
]

all_fk_ok = False
if not export_completed:
    print("Validação de FKs adiada: disponível após uma carga concluída.")
else:
    print(f"VALIDAÇÃO: INTEGRIDADE REFERENCIAL ({len(FK_CHECKS)} consultas)")
    with get_connection() as conn:
        failures = []
        for desc, query in FK_CHECKS:
            orphans = scalar(conn, query)
            if orphans:
                failures.append((desc, orphans))
            print(f"  {desc}: {orphans} órfão(s)")
    all_fk_ok = not failures
    print("FKs OK" if all_fk_ok else f"FALHAS: {failures}")
    if not all_fk_ok:
        raise RuntimeError("Reconciliação de FKs reprovada. Não disponibilize esta carga para análise.")

In [ ]:
# ============================================================
print("=" * 60)
print(f"LegisAnalytica → Supabase | modo={EXPORT_MODE} | {DATA_INICIO} a {DATA_FIM}")
print(f"Tabelas processadas: {len(results)}/{len(TABLES)}")

if EXPORT_MODE == "prepare":
    print("PRONTO PARA REVISÃO: exportação não executada.")
elif EXPORT_MODE == "pilot":
    print("PILOTO FINALIZADO: confira reconciliação, tamanho por tabela e Disk no Supabase; não use o piloto em indicadores.")
elif export_completed and all_match and all_fk_ok:
    print("CARGA INTEGRAL VALIDADA: conteúdo, contagens e FKs aprovados.")
else:
    print("ATENÇÃO: a carga integral não foi validada completamente.")

if export_completed:
    with get_connection() as conn:
        db_mb, wal_mb = storage_metrics(conn)
        print(f"Tamanho banco={db_mb:.1f} MB; WAL={wal_mb:.1f} MB")
    print("Observe também a métrica Disk do projeto no painel do Supabase.")

    with get_connection() as conn:
        for tabela, _, _, _ in TABLES:
            tamanho = scalar(conn, f"SELECT pg_total_relation_size('public.{tabela}')")
            print(f"  {tabela}: {tamanho / 1048576:.2f} MB com índices")
print("Registros existentes são atualizados; chaves extras impedem a carga antes da escrita.")